In [0]:
silver_trucks = spark.table(
    "workspace.transportation_analytics.silver_trucks"
)

silver_truck_utilization = spark.table(
    "workspace.transportation_analytics.silver_truck_utilization_metrics"
)

silver_trucks.printSchema()
silver_truck_utilization.printSchema()

root
 |-- truck_id: string (nullable = true)
 |-- unit_number: integer (nullable = true)
 |-- make: string (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: date (nullable = true)
 |-- acquisition_mileage: integer (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- tank_capacity_gallons: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- home_terminal: string (nullable = true)

root
 |-- truck_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- trips_completed: integer (nullable = true)
 |-- total_miles: integer (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- maintenance_events: integer (nullable = true)
 |-- maintenance_cost: double (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- utilization_rate: double (nullable = true)



In [0]:
from pyspark.sql import functions as F

gold_fleet_utilization = (
    silver_truck_utilization
    .join(
        silver_trucks.select(
            "truck_id",
            "unit_number",
            "make",
            "model_year",
            "fuel_type",
            "status",
            "home_terminal"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_fleet_utilization)

truck_id,month,trips_completed,total_miles,total_revenue,average_mpg,maintenance_events,maintenance_cost,downtime_hours,utilization_rate,unit_number,make,model_year,fuel_type,status,home_terminal
TRK00001,2022-12-01,26,34882,71982.89,6.38,0,0.0,0.0,0.839,3463,Peterbilt,2016,Diesel,Active,Omaha
TRK00001,2023-11-01,29,43564,97542.87,6.51,1,366.3,27.8,0.967,3463,Peterbilt,2016,Diesel,Active,Omaha
TRK00001,2024-08-01,26,35006,74743.74,6.49,1,434.75,31.5,0.839,3463,Peterbilt,2016,Diesel,Active,Omaha
TRK00004,2023-01-01,28,34858,73531.53,6.36,0,0.0,0.0,0.903,3317,Freightliner,2015,Diesel,Active,Salt Lake City
TRK00004,2023-03-01,26,39411,81925.77,6.63,1,428.91,46.1,0.839,3317,Freightliner,2015,Diesel,Active,Salt Lake City
TRK00004,2023-05-01,32,57422,119488.44,6.6,0,0.0,0.0,1.032,3317,Freightliner,2015,Diesel,Active,Salt Lake City
TRK00004,2023-07-01,18,31344,66184.76,6.45,0,0.0,0.0,0.581,3317,Freightliner,2015,Diesel,Active,Salt Lake City
TRK00004,2023-09-01,29,40920,83425.25,6.4,0,0.0,0.0,0.967,3317,Freightliner,2015,Diesel,Active,Salt Lake City
TRK00004,2024-07-01,25,26675,56318.75,6.37,2,842.36,15.5,0.806,3317,Freightliner,2015,Diesel,Active,Salt Lake City
TRK00007,2023-06-01,24,38901,80901.03,6.47,1,1217.74,22.3,0.8,7421,Volvo,2019,Diesel,Active,Charlotte


In [0]:
gold_fleet_utilization = (
    gold_fleet_utilization
    .groupBy(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "fuel_type",
        "status",
        "home_terminal"
    )
    .agg(
        F.sum("trips_completed").alias("total_trips"),
        F.sum("total_miles").alias("total_miles"),
        F.sum("total_revenue").alias("total_revenue"),
        F.avg("utilization_rate").alias("avg_utilization_rate"),
        F.avg("average_mpg").alias("avg_mpg"),
        F.sum("maintenance_cost").alias("total_maintenance_cost"),
        F.sum("downtime_hours").alias("total_downtime_hours")
    )
)

display(gold_fleet_utilization)

truck_id,unit_number,make,model_year,fuel_type,status,home_terminal,total_trips,total_miles,total_revenue,avg_utilization_rate,avg_mpg,total_maintenance_cost,total_downtime_hours
TRK00026,4145,International,2021,Diesel,Active,Miami,895,1252565,2712784.82,0.8155,6.469722222222222,70708.48,787.9
TRK00062,7256,Mack,2017,Diesel,Active,Seattle,909,1273747,2741939.16,0.8290277777777777,6.504166666666666,59646.64,862.1
TRK00070,1854,Kenworth,2015,Diesel,Active,Oklahoma City,943,1319519,2800116.979999999,0.8601388888888888,6.517222222222222,36142.32,307.7
TRK00014,9624,International,2015,Diesel,Active,Nashville,886,1240298,2691574.6799999997,0.8085555555555556,6.519722222222222,71801.82,940.4
TRK00009,2388,International,2015,Diesel,Active,Denver,900,1310170,2832924.8300000005,0.821111111111111,6.5072222222222225,40333.840000000004,522.2999999999998
TRK00058,7026,Mack,2015,Diesel,Active,Columbus,898,1299089,2809911.6000000006,0.8197777777777778,6.482222222222223,45745.439999999995,414.8999999999999
TRK00074,9452,Volvo,2015,Diesel,Active,Indianapolis,881,1298052,2763786.76,0.8050555555555555,6.508333333333334,33044.85,459.19999999999993
TRK00097,7142,Peterbilt,2016,Diesel,Active,Salt Lake City,920,1351692,2891360.4199999995,0.8386944444444444,6.488611111111111,39758.740000000005,431.8000000000001
TRK00001,3463,Peterbilt,2016,Diesel,Active,Omaha,951,1356397,2896338.17,0.8671388888888889,6.488611111111108,51775.98999999999,713.2
TRK00018,6928,International,2015,Diesel,Active,Dallas,859,1194175,2603015.69,0.7836388888888889,6.517222222222222,42430.740000000005,664.5


In [0]:
gold_fleet_utilization = (
    gold_fleet_utilization
    .withColumn(
        "revenue_per_mile",
        F.round(
            F.col("total_revenue") / F.col("total_miles"),
            2
        )
    )
)

display(
    gold_fleet_utilization.select(
        "truck_id",
        "unit_number",
        "total_trips",
        "total_miles",
        "total_revenue",
        "revenue_per_mile",
        "avg_utilization_rate",
        "avg_mpg",
        "total_downtime_hours"
    )
)

truck_id,unit_number,total_trips,total_miles,total_revenue,revenue_per_mile,avg_utilization_rate,avg_mpg,total_downtime_hours
TRK00026,4145,895,1252565,2712784.82,2.17,0.8155,6.469722222222222,787.9
TRK00062,7256,909,1273747,2741939.16,2.15,0.8290277777777777,6.504166666666666,862.1
TRK00070,1854,943,1319519,2800116.979999999,2.12,0.8601388888888888,6.517222222222222,307.7
TRK00014,9624,886,1240298,2691574.6799999997,2.17,0.8085555555555556,6.519722222222222,940.4
TRK00009,2388,900,1310170,2832924.8300000005,2.16,0.821111111111111,6.5072222222222225,522.2999999999998
TRK00058,7026,898,1299089,2809911.6000000006,2.16,0.8197777777777778,6.482222222222223,414.8999999999999
TRK00074,9452,881,1298052,2763786.76,2.13,0.8050555555555555,6.508333333333334,459.19999999999993
TRK00097,7142,920,1351692,2891360.4199999995,2.14,0.8386944444444444,6.488611111111111,431.8000000000001
TRK00001,3463,951,1356397,2896338.17,2.14,0.8671388888888889,6.488611111111108,713.2
TRK00018,6928,859,1194175,2603015.69,2.18,0.7836388888888889,6.517222222222222,664.5


In [0]:
gold_fleet_utilization = (
    gold_fleet_utilization
    .withColumn(
        "revenue_per_mile",
        F.round(
            F.col("total_revenue") / F.col("total_miles"),
            2
        )
    )
)

display(gold_fleet_utilization)

truck_id,unit_number,make,model_year,fuel_type,status,home_terminal,total_trips,total_miles,total_revenue,avg_utilization_rate,avg_mpg,total_maintenance_cost,total_downtime_hours,revenue_per_mile
TRK00026,4145,International,2021,Diesel,Active,Miami,895,1252565,2712784.82,0.8155,6.469722222222222,70708.48,787.9,2.17
TRK00062,7256,Mack,2017,Diesel,Active,Seattle,909,1273747,2741939.16,0.8290277777777777,6.504166666666666,59646.64,862.1,2.15
TRK00070,1854,Kenworth,2015,Diesel,Active,Oklahoma City,943,1319519,2800116.979999999,0.8601388888888888,6.517222222222222,36142.32,307.7,2.12
TRK00014,9624,International,2015,Diesel,Active,Nashville,886,1240298,2691574.6799999997,0.8085555555555556,6.519722222222222,71801.82,940.4,2.17
TRK00009,2388,International,2015,Diesel,Active,Denver,900,1310170,2832924.8300000005,0.821111111111111,6.5072222222222225,40333.840000000004,522.2999999999998,2.16
TRK00058,7026,Mack,2015,Diesel,Active,Columbus,898,1299089,2809911.6000000006,0.8197777777777778,6.482222222222223,45745.439999999995,414.8999999999999,2.16
TRK00074,9452,Volvo,2015,Diesel,Active,Indianapolis,881,1298052,2763786.76,0.8050555555555555,6.508333333333334,33044.85,459.19999999999993,2.13
TRK00097,7142,Peterbilt,2016,Diesel,Active,Salt Lake City,920,1351692,2891360.4199999995,0.8386944444444444,6.488611111111111,39758.740000000005,431.8000000000001,2.14
TRK00001,3463,Peterbilt,2016,Diesel,Active,Omaha,951,1356397,2896338.17,0.8671388888888889,6.488611111111108,51775.98999999999,713.2,2.14
TRK00018,6928,International,2015,Diesel,Active,Dallas,859,1194175,2603015.69,0.7836388888888889,6.517222222222222,42430.740000000005,664.5,2.18


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_fleet_utilization"
)

target.alias("t").merge(
    gold_fleet_utilization.alias("s"),
    "t.truck_id = s.truck_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Fleet Utilization table updated using MERGE")

Gold Fleet Utilization table updated using MERGE


In [0]:
display(
    gold_fleet_utilization
    .orderBy(F.desc("total_revenue"))
    .select(
        "truck_id",
        "unit_number",
        "total_miles",
        "total_revenue",
        "revenue_per_mile"
    )
)

truck_id,unit_number,total_miles,total_revenue,revenue_per_mile
TRK00044,8170,1403922,3045707.5700000003,2.17
TRK00107,8740,1389705,3022073.78,2.17
TRK00041,3658,1404001,3017320.99,2.15
TRK00055,6070,1417530,2997124.4399999995,2.11
TRK00053,9749,1378108,2990269.3800000004,2.17
TRK00015,2531,1386604,2979377.79,2.15
TRK00025,8967,1370297,2942525.2300000004,2.15
TRK00024,6598,1362333,2938674.7,2.16
TRK00057,4178,1377066,2933620.120000001,2.13
TRK00035,9626,1343802,2926212.8200000003,2.18


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.